In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset

import json 

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tqdm import tqdm

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *
from utils import *

from autogluon.tabular import TabularPredictor

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)

notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')

biopsy_df = dfs['biopsy']
model_path = '../models/full10ep_poola_701020_best.pt'
checkpoint = torch.load(model_path, weights_only=False, map_location=device)


all_valid_patient_ids = get_valid_patient_ids(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    min_ts_count=10,
    require_notes=True,
)

with open('../data/splits/pool_assignments.json') as f:
    pool_assignments = json.load(f)
pool_a_ids = np.asarray(pool_assignments['pool_a'])
pool_a_set = set(pool_a_ids.tolist())
selected_patient_ids = np.asarray([pid for pid in all_valid_patient_ids if pid in pool_a_set])

global_split_path = '../data/splits/global_split_pool_a_701020.json'
# Shared global split (70/10/20): fit preprocessing on train only, tune on val, report once on test.
full_split_ids = get_or_create_global_split(
    patient_ids=selected_patient_ids,
    split_json_path=global_split_path,
    train_size=0.7,
    val_size=0.1,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    force_recreate=False,
)

train_dataset = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=full_split_ids['train'],
    fit_preprocessing=True,
    min_ts_count=10,
    require_notes=True,
)
preprocessing_artifacts = train_dataset.preprocessing_artifacts

val_dataset = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=full_split_ids['val'],
    preprocessing_artifacts=preprocessing_artifacts,
    fit_preprocessing=False,
    min_ts_count=10,
    require_notes=True,
)

test_dataset = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=full_split_ids['test'],
    preprocessing_artifacts=preprocessing_artifacts,
    fit_preprocessing=False,
    min_ts_count=10,
    require_notes=True,
)

ts_scaler = train_dataset.ts_scaler
static_scaler = train_dataset.scaler

train_ids = set(full_split_ids['train'])
val_ids = set(full_split_ids['val'])
test_ids = set(full_split_ids['test'])
selected_ids = set(selected_patient_ids.tolist())

assert train_ids.isdisjoint(val_ids), 'Data leakage: train overlaps val'
assert train_ids.isdisjoint(test_ids), 'Data leakage: train overlaps test'
assert val_ids.isdisjoint(test_ids), 'Data leakage: val overlaps test'
assert (train_ids | val_ids | test_ids) == selected_ids, 'Split IDs do not cover selected cohort exactly'

print(f"Eligible Pool A size: {len(selected_patient_ids)}")
print(
    f"Global split sizes (70/10/20): "
    f"{len(train_ids)} / {len(val_ids)} / {len(test_ids)}"
)
print(f"Using shared split file: {global_split_path}")
print('Preprocessing artifacts are fit on train patients only.')
print(f"Train-only preprocessing cardinalities: {train_dataset.categorical_cardinalities}")

In [ ]:
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# vanilla_lstm = VanillaTimeSeriesEncoder()
att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(att_encoder, categorical_cardinalities=train_dataset.categorical_cardinalities, use_static=True, use_notes=True).to(device)
predict_steps_ahead = 1
model.load_state_dict(checkpoint)

In [ ]:
def extract_horizon_reprs(
    dataloader, 
    model, 
    horizons, 
    label_key, 
    rel_days_key=None,
    min_history_days=90,
    max_days=180,
    max_samples_per_patient=100,
    sampling_strategy="uniform",
    is_training_set=False,
    random_state=42
):
    """
    - Feeds each patient's entire time series (minus last step) in one pass.
    - Extracts hidden representations for each time step from model output.
    - For each horizon H, determines if the event occurs within H days from that step.
    - Only includes time steps between min_history_days and max_days.
    - Uses all valid time steps per patient, applying Priority Sampling to preserve positive events.

    Parameters:
    -----------
    dataloader : DataLoader
        The PyTorch dataloader yielding patient data batches
    model : torch.nn.Module
        The trained model to extract representations from
    horizons : list
        List of horizon values (in days) to consider
    label_key : str
        Key in batch dictionary for labels
    rel_days_key : str, optional
        Key for relative days to event (only for single-event labels)
    min_history_days : int, default=90
        Minimum number of days of history required
    max_days : int, default=180
        Maximum number of days to include in the dataset
    max_samples_per_patient : int or None, default=100
        Optional cap per patient per horizon after feature extraction
    sampling_strategy : str, default="uniform"
        One of {"uniform", "random", "first", "last"} when cap is set
    is_training_set : bool, default=False
        If True, enable label-aware priority sampling. Keep False for validation/test.
    random_state : int, default=42
        Random seed used when sampling_strategy is "random"
    """
    from tqdm import tqdm
    import numpy as np
    import torch
    
    device = next(model.parameters()).device
    model.eval()

    # Prepare storage for hidden reps, labels, day-of-step, patient_id
    hr_repr = {H: [] for H in horizons}
    hr_label = {H: [] for H in horizons}
    hr_days  = {H: [] for H in horizons}
    hr_pids  = {H: [] for H in horizons}

    all_pids = set()
    positive_pids = set()

    with torch.no_grad():
        # Add progress bar for the dataloader iteration
        for batch in tqdm(dataloader, desc="Extracting Timesteps"):
            pid  = batch['patient_id']
            slen = batch['seq_len']

            # Static features
            cat_static = batch['static_categorical_features'].to(device)
            num_static = batch['static_numerical_features'].to(device)

            # Time series
            full_ts    = batch['ts_features'].to(device)  # shape (B, T, F)
            timesteps  = batch['timesteps'].to(device)    # shape (B, T)
            mask_      = batch['mask'].to(device)         # shape (B, T)
            value_mask_full = batch['value_mask'].to(device)  # shape (B, T, F)

            # --- Convert single-element Tensors in label_key to float/list ---
            raw_labels_data = batch[label_key]  # shape (B,)
            labels_data = []
            for val in raw_labels_data:
                if isinstance(val, torch.Tensor):
                    if val.numel() == 1:
                        val = float(val.item())
                    else:
                        val = val.cpu().numpy()
                labels_data.append(val)

            # If single-event usage, we also have rel_days_key => shape (B,)
            if rel_days_key and rel_days_key in batch:
                raw_rel_days_data = batch[rel_days_key]
                rel_days_data = []
                for dval in raw_rel_days_data:
                    if isinstance(dval, torch.Tensor):
                        if dval.numel() == 1:
                            dval = float(dval.item())
                        else:
                            dval = dval.cpu().numpy()
                    rel_days_data.append(dval)
                rel_days_data = np.array(rel_days_data)
            else:
                rel_days_data = None  # We'll handle multi-day logic below

            # If using notes
            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps  = batch['notes_timesteps'].to(device)
            notes_mask       = batch['notes_mask'].to(device)

            B, T, F = full_ts.shape

            # Loop over each patient in this batch
            for i in range(B):
                patient_id_i = pid[i]
                all_pids.add(patient_id_i)

                label_or_list = labels_data[i]

                # Keep track of which patients have at least one event
                if isinstance(label_or_list, (int, float, np.number)):
                    if label_or_list == 1:
                        positive_pids.add(patient_id_i)
                elif isinstance(label_or_list, (list, np.ndarray)):
                    if len(label_or_list) > 0:
                        positive_pids.add(patient_id_i)
                elif label_or_list is not None:
                    raise TypeError(f"Unsupported label type: {type(label_or_list)}")

                # If the sequence is too short
                if slen[i] < 2:
                    continue

                # Slice the valid portion of the time series: [0..slen[i]-1]
                seq_len_i = slen[i].item()
                ts_i = full_ts[i:i+1, :seq_len_i, :]   # (1, seq_len_i, F)
                tm_i = timesteps[i:i+1, :seq_len_i]    # (1, seq_len_i)
                mk_i = mask_[i:i+1, :seq_len_i]        # (1, seq_len_i)

                # Model input => omit last step from time series
                inp_seq = ts_i[:, :-1, :]              # (1, seq_len_i-1, F)
                inp_mask = mk_i[:, :-1]
                elapsed_times = build_elapsed_times(tm_i[:, :-1], inp_mask)
                inp_value_mask = value_mask_full[i:i+1, :seq_len_i-1, :]

                notes_emb_i = notes_embeddings[i:i+1]
                notes_ts_i  = notes_timesteps[i:i+1]
                notes_mk_i  = notes_mask[i:i+1]

                # Forward pass
                out, lstm_out, _, _ = model(
                    x=inp_seq,
                    elapsed_times=elapsed_times,     # (1, seq_len_i - 1)
                    timesteps=tm_i[:, :-1],          # (1, seq_len_i - 1)
                    notes_embeddings=notes_emb_i,
                    notes_timesteps=notes_ts_i,
                    static_features=(cat_static[i:i+1], num_static[i:i+1]),
                    mask=inp_mask,                   # (1, seq_len_i - 1)
                    notes_mask=notes_mk_i,
                    value_mask=inp_value_mask
                )
                
                # Time array for all steps
                time_arr = tm_i.cpu().numpy().flatten()  # (seq_len_i,)
                
                # Loop through all valid time steps
                for k in range(seq_len_i - 1):
                    cur_day = time_arr[k]  # Evaluate risk starting on the exact day of the measurement
                    
                    # Skip if outside the desired range
                    if cur_day < min_history_days or cur_day > max_days:
                        continue
                    
                    # Get representation for this time step
                    rep_ = lstm_out[0, k, :].cpu().numpy()
                    
                    # Process each horizon for this valid time step
                    for H in horizons:
                        
                        # If single-event logic is in play
                        if rel_days_data is not None:
                            event_label = label_or_list     # 0 or 1
                            event_day   = rel_days_data[i]  # single day
                            if (event_label == 1) and (0 < (event_day - cur_day) <= H):
                                label_ = 1
                            else:
                                label_ = 0

                        # Else multi-event logic (like rejections)
                        else:
                            if isinstance(label_or_list, (list, np.ndarray)) and len(label_or_list) > 0:
                                label_ = int(any(0 < (d - cur_day) <= H for d in label_or_list))
                            else:
                                label_ = 0

                        # Store every valid step
                        hr_repr[H].append(rep_)
                        hr_label[H].append(label_)
                        hr_days[H].append(cur_day)
                        hr_pids[H].append(patient_id_i)

    # --- THE PRIORITY SAMPLER (Applied after extraction to avoid chronological bias) ---
    print("Applying patient-level caps using Priority Sampling...")
    if max_samples_per_patient is not None and max_samples_per_patient > 0:
        rng = np.random.default_rng(random_state)
        for H in horizons:
            if len(hr_pids[H]) == 0:
                continue

            pids_arr = np.array(hr_pids[H])
            labels_arr = np.array(hr_label[H])
            keep_mask = np.zeros(len(pids_arr), dtype=bool)

            for pid_i in np.unique(pids_arr):
                idx = np.where(pids_arr == pid_i)[0]
                
                if len(idx) <= max_samples_per_patient:
                    chosen = list(idx)
                else:
                    # --- IS_TRAINING_SET == TRUE: PRIORITY SAMPLING ---
                    if is_training_set:
                        pos_idx = idx[labels_arr[idx] == 1]
                        neg_idx = idx[labels_arr[idx] == 0]
                        
                        chosen = []
                        # 1. Grab positive events first
                        if len(pos_idx) > 0:
                            if len(pos_idx) > max_samples_per_patient:
                                chosen.extend(rng.choice(pos_idx, size=max_samples_per_patient, replace=False))
                            else:
                                chosen.extend(pos_idx)
                                
                        # 2. Fill the remaining spots safely
                        remaining_spots = max_samples_per_patient - len(chosen)
                        if remaining_spots > 0 and len(neg_idx) > 0:
                            safe_spots = min(remaining_spots, len(neg_idx))
                            
                            if sampling_strategy == "first":
                                chosen.extend(neg_idx[:safe_spots])
                            elif sampling_strategy == "last":
                                chosen.extend(neg_idx[-safe_spots:])
                            elif sampling_strategy == "uniform":
                                pos = np.linspace(0, len(neg_idx) - 1, num=safe_spots, dtype=int)
                                chosen.extend(neg_idx[pos])
                            elif sampling_strategy == "random":
                                chosen.extend(rng.choice(neg_idx, size=safe_spots, replace=False))
                            else:
                                raise ValueError(f"Unknown sampling_strategy: {sampling_strategy}")
                                
                                
                    else:
                        if sampling_strategy == "first":
                            chosen = list(idx[:max_samples_per_patient])
                        elif sampling_strategy == "last":
                            chosen = list(idx[-max_samples_per_patient:])
                        elif sampling_strategy == "uniform":
                            pos = np.linspace(0, len(idx) - 1, num=max_samples_per_patient, dtype=int)
                            chosen = list(idx[pos])
                        elif sampling_strategy == "random":
                            chosen = list(rng.choice(idx, size=max_samples_per_patient, replace=False))
                        else:
                            raise ValueError(f"Unknown sampling_strategy: {sampling_strategy}")

                keep_mask[np.array(chosen, dtype=int)] = True

            # Apply the mask to filter the dataset
            kept_idx = np.where(keep_mask)[0]
            hr_repr[H] = [hr_repr[H][j] for j in kept_idx]
            hr_label[H] = [hr_label[H][j] for j in kept_idx]
            hr_days[H] = [hr_days[H][j] for j in kept_idx]
            hr_pids[H] = [hr_pids[H][j] for j in kept_idx]

    # Convert lists to numpy arrays for downstream compatibility
    for H in horizons:
        hr_repr[H] = np.array(hr_repr[H])
        hr_label[H] = np.array(hr_label[H])
        hr_days[H]  = np.array(hr_days[H])
        hr_pids[H]  = np.array(hr_pids[H])

    # Calculate and print final statistics
    for H in horizons:
        unique_pids, counts = np.unique(hr_pids[H], return_counts=True) if len(hr_pids[H]) else (np.array([]), np.array([]))
        total_patients = len(unique_pids)
        avg_samples = float(np.mean(counts)) if len(counts) else 0.0
        max_samples = int(np.max(counts)) if len(counts) else 0
        
        print(f"Horizon {H}: {total_patients} patients, avg {avg_samples:.1f} samples/patient, max {max_samples} samples/patient")

    print(f"Total unique patients processed: {len(all_pids)}, patients with at least one event: {len(positive_pids)}")
    return hr_repr, hr_label, hr_days, hr_pids

In [ ]:
horizons = [30, 90, 180]
min_history_days = 90
max_days = 720
max_samples_per_patient = 100  # set to None for no cap
sampling_strategy = "uniform"

print("Embedding patients...")
print("Using is_training_set=True for train (priority sampling preserves positive steps), False for val/test.")

train_graft_repr, train_graft_lbl, train_graft_days, train_graft_pids = extract_horizon_reprs(train_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days, max_samples_per_patient=max_samples_per_patient, sampling_strategy=sampling_strategy, is_training_set=True)
val_graft_repr, val_graft_lbl, val_graft_days, val_graft_pids = extract_horizon_reprs(val_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days, max_samples_per_patient=max_samples_per_patient, sampling_strategy=sampling_strategy, is_training_set=False)
test_graft_repr, test_graft_lbl, test_graft_days, test_graft_pids = extract_horizon_reprs(test_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days, max_samples_per_patient=max_samples_per_patient, sampling_strategy=sampling_strategy, is_training_set=False)

train_rej_repr, train_rej_lbl, train_rej_days, train_rej_pids = extract_horizon_reprs(train_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days, max_samples_per_patient=max_samples_per_patient, sampling_strategy=sampling_strategy, is_training_set=True)
val_rej_repr, val_rej_lbl, val_rej_days, val_rej_pids = extract_horizon_reprs(val_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days, max_samples_per_patient=max_samples_per_patient, sampling_strategy=sampling_strategy, is_training_set=False)
test_rej_repr, test_rej_lbl, test_rej_days, test_rej_pids = extract_horizon_reprs(test_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days, max_samples_per_patient=max_samples_per_patient, sampling_strategy=sampling_strategy, is_training_set=False)

train_mort_repr, train_mort_lbl, train_mort_days, train_mort_pids = extract_horizon_reprs(train_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days, max_samples_per_patient=max_samples_per_patient, sampling_strategy=sampling_strategy, is_training_set=True)
val_mort_repr, val_mort_lbl, val_mort_days, val_mort_pids = extract_horizon_reprs(val_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days, max_samples_per_patient=max_samples_per_patient, sampling_strategy=sampling_strategy, is_training_set=False)
test_mort_repr, test_mort_lbl, test_mort_days, test_mort_pids = extract_horizon_reprs(test_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days, max_samples_per_patient=max_samples_per_patient, sampling_strategy=sampling_strategy, is_training_set=False)

# Print some stats about the extracted data
for H in horizons:
    print(f"\n===== Extracted Features for Horizon {H} days =====")
    print(f"Graft Loss Train/Val/Test: {train_graft_repr[H].shape} / {val_graft_repr[H].shape} / {test_graft_repr[H].shape}")
    print(f"Rejection Train/Val/Test: {train_rej_repr[H].shape} / {val_rej_repr[H].shape} / {test_rej_repr[H].shape}")
    print(f"Mortality Train/Val/Test: {train_mort_repr[H].shape} / {val_mort_repr[H].shape} / {test_mort_repr[H].shape}")

In [ ]:
### LOGISTIC REGRESSION - This is just a sanity check to see if the extracted features have signal for the downstream tasks. We can replace this with more complex models later if desired.

def train_and_eval_logistic(X_train, y_train, X_test, y_test, event_name="Event"):
    clf = LogisticRegression(class_weight={0:1, 1:10},max_iter=1000).fit(X_train, y_train)
    y_pred_proba = clf.predict_proba(X_test)[:, 1]
    
    threshold = .2
    y_pred = (y_pred_proba >= threshold).astype(int)

    #y_pred = clf.predict(X_test)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"\n{event_name} Prediction:")
    print(f"AUC = {roc_auc_score(y_test, y_pred_proba):.4f}")
    print(f"Acc = {accuracy_score(y_test, y_pred):.4f}")
    print(f"Prec = {precision_score(y_test, y_pred):.4f}")
    print(f"Recall (Sensitivity) = {recall_score(y_test, y_pred):.4f}")
    print(f"Specificity = {specificity:.4f}")
    print(f"F1 = {f1_score(y_test, y_pred):.4f}")

for H in horizons:
    train_and_eval_logistic(train_graft_repr[H], train_graft_lbl[H], test_graft_repr[H], test_graft_lbl[H], event_name=f"GraftLoss@{H}")
    #train_and_eval_logistic(train_mort_repr[H], train_mort_lbl[H], test_mort_repr[H], test_mort_lbl[H], event_name=f"Mortality@{H}")
    #train_and_eval_logistic(train_rej_repr[H], train_rej_lbl[H], test_rej_repr[H],  test_rej_lbl[H], event_name=f"Rejection@{H}")


In [ ]:
def train_and_eval_mlp_ckpt(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    event_name="Event",
    epochs=30,
    batch_size=32,
    lr=5e-3,
    use_upsampling=True,
    eval_interval=1,
    min_checkpoint_epoch=1,
):
    """
    MLP training for a fixed number of epochs; checkpoint on validation AUC only,
    then evaluate the selected checkpoint once on the untouched test set.
    """
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_t   = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_val_t   = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to(device)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).to(device)

    train_ds = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model_mlp = SimpleMLP(input_dim=X_train.shape[1]).to(device)
    optimizer = optim.Adam(model_mlp.parameters(), lr=lr, weight_decay=1e-4)

    if use_upsampling:
        num_pos = max(1, np.sum(y_train == 1))
        num_neg = np.sum(y_train == 0)
        dynamic_weight = num_neg / num_pos
        pos_weight = torch.tensor([dynamic_weight], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        print(f"[{event_name}] pos_weight={dynamic_weight:.3f} (neg={num_neg}, pos={num_pos})")
    else:
        criterion = nn.BCEWithLogitsLoss()

    def predict_proba(model_eval, X_eval_t):
        model_eval.eval()
        with torch.no_grad():
            logits = model_eval(X_eval_t)
            probs = torch.sigmoid(logits).cpu().numpy().reshape(-1)
        return probs

    def select_threshold(y_true, probs):
        y_true = np.asarray(y_true).astype(int)
        candidate_thresholds = np.unique(np.concatenate([
            np.linspace(0.05, 0.95, 19),
            probs,
        ]))
        best_threshold = 0.5
        best_f1 = -1.0
        for threshold in candidate_thresholds:
            pred = (probs >= threshold).astype(int)
            f1 = f1_score(y_true, pred, zero_division=0)
            if f1 > best_f1 or (np.isclose(f1, best_f1) and abs(threshold - 0.5) < abs(best_threshold - 0.5)):
                best_f1 = f1
                best_threshold = float(threshold)
        return best_threshold

    def evaluate_split(y_true, probs, threshold):
        true_np = np.asarray(y_true).astype(int)
        pred = (probs >= threshold).astype(int)

        auc = roc_auc_score(true_np, probs) if len(set(true_np)) > 1 else float('nan')
        acc = accuracy_score(true_np, pred)
        prec = precision_score(true_np, pred, zero_division=0)
        rec = recall_score(true_np, pred, zero_division=0)
        f1 = f1_score(true_np, pred, zero_division=0)
        tn, fp, fn, tp = confusion_matrix(true_np, pred).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        return {
            'auc': auc,
            'acc': acc,
            'prec': prec,
            'recall': rec,
            'spec': spec,
            'f1': f1,
        }

    best_val_auc = -1.0
    best_state = None
    best_epoch = 0
    history = []

    ckpt_path = f'../models/{event_name}_clf.pth'
    best_threshold = 0.5

    for epoch in range(1, epochs + 1):
        model_mlp.train()
        total_loss = 0.0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model_mlp(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(train_loader))

        should_eval = (epoch % eval_interval == 0) or (epoch == epochs)
        if should_eval:
            val_probs = predict_proba(model_mlp, X_val_t)
            threshold = select_threshold(y_val, val_probs)
            val_metrics = evaluate_split(y_val, val_probs, threshold)
            val_metrics['epoch'] = epoch
            val_metrics['loss'] = avg_loss
            val_metrics['threshold'] = threshold
            history.append(val_metrics)

            can_checkpoint = epoch >= min_checkpoint_epoch
            if can_checkpoint and not np.isnan(val_metrics['auc']) and val_metrics['auc'] > best_val_auc:
                best_val_auc = val_metrics['auc']
                best_state = {k: v.detach().cpu().clone() for k, v in model_mlp.state_dict().items()}
                best_epoch = epoch
                best_threshold = threshold
                torch.save(best_state, ckpt_path)
                suffix = ' *best*'
            else:
                suffix = ''

            print(
                f"[{event_name}] Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}, "
                f"Val AUC: {val_metrics['auc']:.4f}, Val F1: {val_metrics['f1']:.4f}, Thr: {threshold:.3f}{suffix}"
            )

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model_mlp.state_dict().items()}
        best_epoch = epochs
        best_threshold = select_threshold(y_val, predict_proba(model_mlp, X_val_t))
        torch.save(best_state, ckpt_path)
        print(f"[{event_name}] No eligible validation checkpoint met criteria; saved final epoch {epochs}.")

    model_mlp.load_state_dict(best_state)
    test_probs = predict_proba(model_mlp, X_test_t)
    test_metrics = evaluate_split(y_test, test_probs, best_threshold)
    test_metrics['epoch'] = best_epoch
    test_metrics['threshold'] = best_threshold

    print(
        f"[{event_name}] Finished full {epochs} epochs. "
        f"Best epoch={best_epoch} by val AUC={best_val_auc:.4f} | "
        f"Test AUC={test_metrics['auc']:.4f}, Test F1={test_metrics['f1']:.4f}, Thr={best_threshold:.3f}\n"
    )

    return test_metrics, history

In [ ]:
results = {}
for H in horizons:
    print(f"\n{'='*60}")
    print(f"Horizon: {H} days")
    print(f"{'='*60}")
    
    m, _ = train_and_eval_mlp_ckpt(
        X_train=train_graft_repr[H], y_train=train_graft_lbl[H],
        X_val=val_graft_repr[H], y_val=val_graft_lbl[H],
        X_test=test_graft_repr[H], y_test=test_graft_lbl[H],
        event_name=f"GraftLoss@{H}", epochs=30, batch_size=32, lr=5e-3,
        use_upsampling=True, eval_interval=1, min_checkpoint_epoch=1
    )
    results[f"GraftLoss@{H}"] = m

    m, _ = train_and_eval_mlp_ckpt(
        X_train=train_rej_repr[H], y_train=train_rej_lbl[H],
        X_val=val_rej_repr[H], y_val=val_rej_lbl[H],
        X_test=test_rej_repr[H], y_test=test_rej_lbl[H],
        event_name=f"Rejection@{H}", epochs=30, batch_size=32, lr=5e-3,
        use_upsampling=True, eval_interval=1, min_checkpoint_epoch=1
    )
    results[f"Rejection@{H}"] = m

    m, _ = train_and_eval_mlp_ckpt(
        X_train=train_mort_repr[H], y_train=train_mort_lbl[H],
        X_val=val_mort_repr[H], y_val=val_mort_lbl[H],
        X_test=test_mort_repr[H], y_test=test_mort_lbl[H],
        event_name=f"Mortality@{H}", epochs=30, batch_size=32, lr=5e-3,
        use_upsampling=True, eval_interval=1, min_checkpoint_epoch=1
    )
    results[f"Mortality@{H}"] = m

# Print summary table
print(f"\n{'='*60}")
print("SUMMARY — Test metrics from best validation checkpoint (30 epochs)")
print(f"{'='*60}")
print(f"{'Event':<20} {'30-day':>10} {'90-day':>10} {'180-day':>10}")
for event in ['GraftLoss', 'Rejection', 'Mortality']:
    aucs = [results[f"{event}@{H}"]['auc'] for H in [30, 90, 180]]
    print(f"{event:<20} {aucs[0]:>10.4f} {aucs[1]:>10.4f} {aucs[2]:>10.4f}")
print(f"\nAll results dict: {json.dumps({k: {mk: round(mv, 4) for mk, mv in v.items()} for k, v in results.items()}, indent=2)}")

In [ ]:
# Print compact summary
for event in ['GraftLoss', 'Rejection', 'Mortality']:
    aucs = [results[f"{event}@{H}"]['auc'] for H in [30, 90, 180]]
    epochs = [results[f"{event}@{H}"]['epoch'] for H in [30, 90, 180]]
    print(f"{event}: 30d={aucs[0]:.4f}(ep{epochs[0]}), 90d={aucs[1]:.4f}(ep{epochs[1]}), 180d={aucs[2]:.4f}(ep{epochs[2]})")

# Also print all metrics for the MD file
for k, v in results.items():
    print(f"\n{k}: AUC={v['auc']:.4f}, Acc={v['acc']:.4f}, Prec={v['prec']:.4f}, Recall={v['recall']:.4f}, Spec={v['spec']:.4f}, F1={v['f1']:.4f}, BestEpoch={v['epoch']}")